In [0]:
# ------------------------------------------------------------------------------
# 3. Register: udf_compute_quality_score
# ------------------------------------------------------------------------------
spark.sql("""
CREATE OR REPLACE FUNCTION capstone_project.capstone_schema.udf_compute_quality_score(
    customer_id STRING, product_id STRING, quantity INT, unit_price FLOAT, discount FLOAT
)
RETURNS FLOAT
LANGUAGE PYTHON
COMMENT 'Calculates a 0.0-1.0 score based on nulls, ranges, and referential integrity'
AS $$
    score = 1.0
    if not customer_id: score -= 0.2
    if not product_id: score -= 0.2
    if quantity is None or quantity < 1: score -= 0.3
    if unit_price is None or unit_price < 0.0: score -= 0.3
    if discount is None: score -= 0.1
    elif discount < 0.0 or discount > 1.0: score -= 0.2
    return max(0.0, round(score, 2))
$$
""")

# ------------------------------------------------------------------------------
# 4. Apply Governance 
# ------------------------------------------------------------------------------
print("Applying Unity Catalog execution grants...")

spark.sql("GRANT EXECUTE ON FUNCTION capstone_project.capstone_schema.udf_compute_quality_score TO `account users`")

print("Success! UDFs are registered to capstone_project.capstone_schema")